# 04 · Cálculo directo por hoja

## Objetivo
Evaluar la cadena de fluido y el balance de cada hoja, y comparar PIP y THD con la caché de la misma hoja.

## Alcance
La comparación usa la unidad del rótulo: pies para PIP y THD. No hay una segunda medición de PIP. Un caso con BS&W vacío se deja sin resultado.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

from ingesta.leer_bes_jun import leer_libro
from modelo_deterministico.fluidos_bes_jun import evaluar_fluido
from modelo_deterministico.balance_bes_jun import evaluar_balance
hojas = leer_libro(libro)
for hoja in hojas:
    fluido = evaluar_fluido(hoja["api_deg"], hoja["temperatura_c"], hoja["bsw_numero"], hoja["densidad_agua_lb_gal"])
    grad = None if fluido.estado != "calculable" else fluido.resultados["gradiente_psi_ft"]["valor"]
    balance = evaluar_balance(
        hoja["presion_sensor_psi"], hoja["thp_psi"], hoja["tvd_intake_ft"], hoja["tvd_sensor_ft"],
        hoja["tvd_descarga_ft"], grad, hoja["caudal_operacion_bfpd"], hoja["id_tubing_in"],
    )
    pip = balance.resultados["pip_ft"]["valor"]
    thd = balance.resultados["thd_total_ft"]["valor"]
    print(
        hoja["hoja"], hoja["pozo"],
        "fluido", fluido.estado, "balance", balance.estado,
        "diff_pip_ft", abs(pip - hoja["cache_pip_ft"]),
        "diff_thd_ft", abs(thd - hoja["cache_thd_ft"]),
    )
    assert balance.estado == "calculable"
    assert abs(pip - hoja["cache_pip_ft"]) < 1e-6
base = hojas[0]
incompleto = evaluar_fluido(base["api_deg"], base["temperatura_c"], None, base["densidad_agua_lb_gal"])
print("caso_construido", base["hoja"], incompleto.estado, incompleto.motivo)
assert incompleto.estado == "no_calculable"


QF-554H QF-084 fluido calculable balance calculable diff_pip_ft 0.0 diff_thd_ft 9.094947017729282e-13
QF-545H QF-301 fluido calculable balance calculable diff_pip_ft 0.0 diff_thd_ft 4.547473508864641e-13
QF-500H QF-147 fluido calculable balance calculable diff_pip_ft 0.0 diff_thd_ft 4.547473508864641e-13
QF-378HST QF-005 fluido calculable balance calculable diff_pip_ft 0.0 diff_thd_ft 4.547473508864641e-13
QF-549HST QF-306 fluido calculable balance calculable diff_pip_ft 0.0 diff_thd_ft 0.0
caso_construido QF-554H no_calculable Faltan entradas esenciales del fluido: bsw_numero


## Entradas y criterio
Por hoja se usan API en °API, temperatura de fondo en °C, el número de BS&W, la densidad de agua, la presión de sensor y el THP en psi, las TVD en ft, el caudal de operación en BFPD y el diámetro interno en pulgadas. El gradiente sale de la cadena de fluido de esa hoja y entra al balance.

La aserción exige que el PIP calculado y el PIP cacheado difieran en menos de 1e-6 ft. La salida también imprime la diferencia del THD. En las cinco hojas la diferencia de PIP es 0 ft. La de THD es 0 ft en QF-549HST, 9.09e-13 ft en QF-554H y 4.55e-13 ft en QF-545H, QF-500H y QF-378HST. Es resto de punto flotante, no un desvío de medición.

## Caso sin resultado
Se repite QF-554H con BS&W vacío. El estado es `no_calculable` y no se inventa el número. Coincidir con la caché reproduce la hoja. No contrasta el PIP con otro sensor y no predice un día posterior.
